In [17]:
import sys
import asyncio
import json
from concurrent.futures import ThreadPoolExecutor
from bs4 import BeautifulSoup
from camoufox.sync_api import Camoufox
from curl_cffi import requests

### Get user agent and cookies

In [ ]:
# Fix Windows asyncio subprocess policy so threads can spawn the browser
# if sys.platform == "win32":
#     asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

target_url = "https://www.medimops.de"

# Helper function running pure synchronous Camoufox
def fetch_cloudflare_tokens(url):
    print("Solving Cloudflare challenge in background thread...")
    with Camoufox(headless=True) as browser:
        page = browser.new_page()
        page.goto(url, wait_until="networkidle")
        
        cookies = {c['name']: c['value'] for c in page.context.cookies()}
        user_agent = page.evaluate("navigator.userAgent")
        
    return cookies, user_agent

# Step 1: Execute synchronous Camoufox isolated in a background thread
with ThreadPoolExecutor(max_workers=1) as executor:
    future = executor.submit(fetch_cloudflare_tokens, target_url)
    cookies, user_agent = future.result()

Solving Cloudflare challenge in background thread...


## Scrape movie prices

In [83]:
def scrape_movie_prices(session: requests.Session, movie_title="Conclave"):
    """
    """
    # Add movie title + "blu ray" as search parameter
    params = {
            "searchparam" : f"{movie_title} blu ray",
            "fcIsSearch" : 1
            }
    
    # Query for movie data
    print("[INFO] Querying movie data...")
    response = session.get(
        # Use "Blu-ray" url to only get results for blu rays (no books, DVDs etc.)
        url="https://www.medimops.de/Blu-ray,/?condition=UsedVeryGood%2CUsedGood%2CNewItem%2CLibriNew",
        params=params
        )

    # Extract the product data from full SSR HTML
    print("[INFO] Parsing soup...")
    soup = BeautifulSoup(response.text, "html.parser")

    # ------------------------------------------------------------------------------------
    found_titles = soup.find_all(class_="ProductSearch_title__RhpT3")
    print(f"[INFO] Found {len(found_titles)} titles!")

    # ------------------------------------------------------------------------------------
    found_pricing = soup.find_all(class_="ProductSearch_price__N58Fa")
    print(f"[INFO] Found {len(found_pricing)} prices!")

    if len(found_pricing)/len(found_titles) != 2:
        print("[WARNING] Number of prices does not match number of titles!")

    # ------------------------------------------------------------------------------------
    def _get_price(elem):
        txt = (
            elem.text
            .replace("\xa0", " ")
            .replace("€","")
            .replace(",",".")
            .strip()
        )
        return float(txt)

    found_pricing_clean = [(_get_price(c), _get_price(p)) for c, p in zip(found_pricing[::2], found_pricing[1::2])]

    # ------------------------------------------------------------------------------------
    found_conditions = soup.find_all(class_="ProductSearch_condition__810YM ProductSearch_flexColumn__HBtVh mobile-only")
    print(f"[INFO] Found {len(found_conditions)} conditions!")

    if len(found_titles) != len(found_conditions):
        print("[WARNING] Number of conditions does not match number of titles!")

    found_conditions = [cond.text for cond in found_conditions]

    # ------------------------------------------------------------------------------------
    results = [(t.text, p[0], p[1], c) for t,p,c in zip(found_titles, found_pricing_clean, found_conditions)]

    return results

### Create session object

Create a session object to reuse TLP/TCP connections etc. accross multiple requests

In [64]:
# Create a persistent session to hold tokens across requests
session = requests.Session()

session.headers.update({
    "User-Agent": user_agent,
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "de-DE,de;q=0.9,en-US;q=0.8,en;q=0.7",
})

session.cookies.update(cookies)
session.impersonate = "chrome124"  # Best match for Camoufox

### Query page

In [ ]:
# --------------------------------------------------------------------------------
movie_title = "No country for old men"

scrape_movie_prices(session, movie_title=movie_title)

[INFO] Querying movie data...
[INFO] Parsing soup...
[INFO] Found 11 titles!
[INFO] Found 22 prices!
[INFO] Found 11 conditions!


[('No Country For Old Men [Blu-ray]', 5.49, 5.49, 'Gebraucht - Sehr gut'),
 ('No Country for Old Men [Blu-ray]', 7.79, 7.79, 'Gebraucht - Sehr gut'),
 ('No Country For Old Men (Limitierte Steelbook Edition) [Blu-ray]',
  17.49,
  17.49,
  'Gebraucht - Sehr gut'),
 ('No Country For Old Men [Blu-ray] [UK Import]',
  9.59,
  9.59,
  'Gebraucht - Sehr gut'),
 ('No country for old men [Blu-ray]', 12.59, 12.59, 'Gebraucht - Sehr gut'),
 ('The Old Way [Blu-ray]', 6.69, 6.69, 'Gebraucht - Sehr gut'),
 ('OLD [Blu-ray]', 6.49, 6.49, 'Gebraucht - Gut'),
 ('OldBoy [Blu-ray]', 7.49, 7.49, 'Gebraucht - Sehr gut'),
 ('Oldboy [Blu-ray]', 7.49, 7.49, 'Gebraucht - Sehr gut'),
 ('Lang lebe Charlie Countryman [Blu-ray]',
  3.49,
  3.49,
  'Gebraucht - Sehr gut'),
 ('Bad Country  (inkl. Digital Ultraviolet) [Blu-ray]',
  3.79,
  3.79,
  'Gebraucht - Sehr gut')]